In [ ]:
import pandas as pd

# Fungsi untuk merapikan 1 file (satu tahun)
def bersihkan_satu_tahun(nama_file, tahun):
    # baca file
    df_raw = pd.read_csv(nama_file)

    # nama kolom di file BPS
    col_prov = df_raw.columns[0]           # "38 Provinsi"
    age_cols = df_raw.columns[1:5]         # 4 kolom umur (7-12, 13-15, 16-18, 19-23)

    # ambil baris yang ada nama provinsinya
    df = df_raw[df_raw[col_prov].notna()].copy()

    # buang baris "INDONESIA" (agregat nasional)
    df = df[df[col_prov] != "INDONESIA"]

    # ubah dari wide ke long: 4 kolom umur -> 1 kolom
    df_long = df.melt(
        id_vars=[col_prov],
        value_vars=age_cols,
        var_name="KolomAsal",
        value_name="Presentase"
    )

    # mapping nama kolom ke umur
    umur_map = {
        age_cols[0]: "7-12",
        age_cols[1]: "13-15",
        age_cols[2]: "16-18",
        age_cols[3]: "19-23",
    }
    df_long["Umur"] = df_long["KolomAsal"].map(umur_map)

    # tambah kolom tahun
    df_long["Tahun"] = tahun

    # rapikan nama kolom
    df_long.rename(columns={col_prov: "Provinsi"}, inplace=True)

    # pastikan presentase numerik, "-" jadi NaN
    df_long["Presentase"] = pd.to_numeric(df_long["Presentase"], errors="coerce")

    # pilih urutan kolom yang diinginkan
    df_long = df_long[["Provinsi", "Umur", "Tahun", "Presentase"]]
    return df_long

# ==== PANGGIL FUNGSI UNTUK 5 TAHUN ====
file_per_tahun = {
    2020: "Angka Partisipasi Sekolah (APS) Menurut Provinsi dan Kelompok Umur, 2020.csv",
    2021: "Angka Partisipasi Sekolah (APS) Menurut Provinsi dan Kelompok Umur, 2021.csv",
    2022: "Angka Partisipasi Sekolah (APS) Menurut Provinsi dan Kelompok Umur, 2022.csv",
    2023: "Angka Partisipasi Sekolah (APS) Menurut Provinsi dan Kelompok Umur, 2023.csv",
    2024: "Angka Partisipasi Sekolah (APS) Menurut Provinsi dan Kelompok Umur, 2024.csv",
}

df_list = []
for thn, nama_file in file_per_tahun.items():
    df_list.append(bersihkan_satu_tahun(nama_file, thn))

# gabung semua tahun
df_all = pd.concat(df_list, ignore_index=True)

# atur urutan umur agar 7-12, 13-15, 16-18, 19-23 (bukan urut alfabet)
umur_order = ["7-12", "13-15", "16-18", "19-23"]
df_all["Umur"] = pd.Categorical(df_all["Umur"], categories=umur_order, ordered=True)

# urutkan: Provinsi -> Umur -> Tahun (tahun terbaru dulu)
df_all = df_all.sort_values(["Provinsi", "Umur", "Tahun"],
                            ascending=[True, True, False])

# lihat 20 baris pertama
df_all.head(20)


,Provinsi,Umur,Tahun,Presentase
608,ACEH,7-12,2024,99.42
456,ACEH,7-12,2023,99.43
304,ACEH,7-12,2022,99.44
152,ACEH,7-12,2021,99.67
0,ACEH,7-12,2020,99.84
646,ACEH,13-15,2024,97.77
494,ACEH,13-15,2023,97.72
342,ACEH,13-15,2022,97.96
190,ACEH,13-15,2021,98.42
38,ACEH,13-15,2020,98.49


In [ ]:
df_all.to_csv("APS_Provinsi_Umur_2020_2024_rapi.csv", index=False)


In [ ]:
import pandas as pd

# Fungsi untuk merapikan 1 file IPM (satu tahun)
def bersihkan_ipm(nama_file, tahun):
    # baca CSV
    df_raw = pd.read_csv(nama_file)

    # ambil dua kolom pertama: Provinsi & IPM
    col_prov = df_raw.columns[0]
    col_ipm  = df_raw.columns[1]

    df = df_raw[[col_prov, col_ipm]].copy()

    # buang baris kosong & agregat nasional
    df = df[df[col_prov].notna()]
    df = df[df[col_prov] != "INDONESIA"]

    # ganti nama kolom
    df.rename(columns={col_prov: "Provinsi", col_ipm: "IPM_raw"}, inplace=True)

    # --- bereskan nilai IPM ---
    # ubah ke string, ganti koma -> titik
    s = (
        df["IPM_raw"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    # ubah ke numerik
    df["IPM"] = pd.to_numeric(s, errors="coerce")

    # kalau ada yang >100 (misal 7470) anggap perlu dibagi 100 -> 74.70
    df.loc[df["IPM"] > 100, "IPM"] = df.loc[df["IPM"] > 100, "IPM"] / 100

    # tambah kolom tahun
    df["Tahun"] = tahun

    # pilih urutan kolom yang diinginkan
    df = df[["Provinsi", "Tahun", "IPM"]]
    return df

# ===== PANGGIL FUNGSI UNTUK 4 TAHUN =====
file_per_tahun = {
    2020: "Indeks Pembangunan Manusia Menurut Provinsi, 2020.csv",
    2021: "Indeks Pembangunan Manusia Menurut Provinsi, 2021.csv",
    2022: "Indeks Pembangunan Manusia Menurut Provinsi, 2022.csv",
    2023: "Indeks Pembangunan Manusia Menurut Provinsi, 2023.csv",
}

df_list = []
for thn, nama_file in file_per_tahun.items():
    df_list.append(bersihkan_ipm(nama_file, thn))

# gabungkan semua tahun
df_ipm = pd.concat(df_list, ignore_index=True)

# urutkan: Provinsi (A–Z), Tahun (terbaru dulu)
df_ipm = df_ipm.sort_values(["Provinsi", "Tahun"],
                            ascending=[True, False])

# lihat 20 baris pertama
df_ipm.head(20)


,Provinsi,Tahun,IPM
165,</i>,2023,NaN
164,</i>Indeks Pembangunan Manusia (IPM) 2020-2023...,2023,NaN
123,Aceh,2023,74.70
82,Aceh,2022,74.11
41,Aceh,2021,73.48
0,Aceh,2020,73.29
139,Bali,2023,78.01
98,Bali,2022,77.40
57,Bali,2021,76.69
16,Bali,2020,76.52


In [ ]:
df_ipm.to_csv("IPM_Provinsi_2020_2023_rapi.csv",
              index=False,
              decimal=',')   # simpan pakai koma, misal 74,70


In [ ]:
import pandas as pd

# Fungsi untuk merapikan 1 file (satu tahun)
def bersihkan_telepon(nama_file, tahun):
    # baca CSV; BPS biasanya pakai koma sebagai desimal
    df_raw = pd.read_csv(nama_file, decimal=',')

    # kolom: [0] = Provinsi, [1] = Perkotaan, [2] = Perdesaan
    col_prov = df_raw.columns[0]
    seg_cols = df_raw.columns[1:3]

    # ambil kolom penting saja
    df = df_raw[[col_prov] + list(seg_cols)].copy()

    # buang baris kosong & agregat nasional
    df = df[df[col_prov].notna()]
    df = df[df[col_prov] != "INDONESIA"]

    # ubah dari wide (Perkotaan, Perdesaan) -> long (Segmentasi Wilayah)
    df_long = df.melt(
        id_vars=[col_prov],
        value_vars=seg_cols,
        var_name="Segmentasi Wilayah",
        value_name="Presentase"
    )

    # rapikan nama kolom
    df_long.rename(columns={col_prov: "Provinsi"}, inplace=True)

    # samakan format nama provinsi (UPPERCASE biar konsisten dengan dataset lain)
    df_long["Provinsi"] = df_long["Provinsi"].str.upper()

    # pastikan Presentase numerik
    df_long["Presentase"] = pd.to_numeric(df_long["Presentase"], errors="coerce")

    # tambah tahun
    df_long["Tahun"] = tahun

    # urutan kolom final
    df_long = df_long[["Provinsi", "Segmentasi Wilayah", "Tahun", "Presentase"]]
    return df_long


# ===== panggil untuk tiap tahun =====
file_per_tahun = {
    2020: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2020.csv",
    2021: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2021.csv",
    2022: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2022.csv",
    2023: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2023.csv",
    2024: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2024.csv",
}

df_list = []
for thn, nama_file in file_per_tahun.items():
    df_list.append(bersihkan_telepon(nama_file, thn))

# gabung semua tahun
df_telpon = pd.concat(df_list, ignore_index=True)

# atur urutan Segmentasi: Perkotaan dulu baru Perdesaan
seg_order = ["Perkotaan", "Perdesaan"]
df_telpon["Segmentasi Wilayah"] = pd.Categorical(
    df_telpon["Segmentasi Wilayah"],
    categories=seg_order,
    ordered=True
)

# urutkan: Provinsi -> Segmentasi Wilayah -> Tahun (tahun terbaru dulu)
df_telpon = df_telpon.sort_values(
    ["Provinsi", "Segmentasi Wilayah", "Tahun"],
    ascending=[True, True, False]
)

# cek 20 baris pertama (harus mirip screenshot: ACEH Perkotaan 2024-2020, lalu Perdesaan 2024-2020)
df_telpon.head(20)


,Provinsi,Segmentasi Wilayah,Tahun,Presentase
304,ACEH,NaN,2024,0.52
342,ACEH,NaN,2024,0.26
228,ACEH,NaN,2023,0.41
266,ACEH,NaN,2023,0.37
152,ACEH,NaN,2022,0.78
190,ACEH,NaN,2022,0.13
76,ACEH,NaN,2021,0.74
114,ACEH,NaN,2021,0.18
0,ACEH,NaN,2020,0.41
38,ACEH,NaN,2020,0.12


In [ ]:
df_telpon.to_csv(
    "Telepon_Tetap_Provinsi_Segmentasi_2020_2024_rapi.csv",
    index=False,
    decimal=','   # simpan pakai koma, misal 0,52
)


In [ ]:
import pandas as pd

def bersihkan_telepon(nama_file, tahun):
    # Baca CSV BPS (koma sebagai desimal)
    df_raw = pd.read_csv(nama_file, decimal=',')

    # Kolom pertama = nama provinsi
    col_prov = df_raw.columns[0]

    # Cari kolom yang mengandung "Perkotaan" atau "Perdesaan"
    seg_cols = []
    for c in df_raw.columns[1:]:
        cs = str(c)
        # abaikan kolom gabungan Perkotaan+Perdesaan
        if "Perkotaan+Perdesaan" in cs:
            continue
        if "Perkotaan" in cs or "Perdesaan" in cs:
            seg_cols.append(c)

    # Ambil hanya Provinsi + kolom segmen
    df = df_raw[[col_prov] + seg_cols].copy()

    # Buang baris kosong & agregat nasional
    df = df[df[col_prov].notna()]
    df = df[df[col_prov] != "INDONESIA"]

    # Wide -> Long
    df_long = df.melt(
        id_vars=[col_prov],
        value_vars=seg_cols,
        var_name="KolomAsal",
        value_name="Presentase"
    )

    # Rapikan nama provinsi
    df_long.rename(columns={col_prov: "Provinsi"}, inplace=True)
    df_long["Provinsi"] = df_long["Provinsi"].str.upper()

    # Mapping KolomAsal -> Segmentasi Wilayah
    def map_segmentasi(s):
        s = str(s)
        if "Perkotaan" in s:
            return "Perkotaan"
        if "Perdesaan" in s:
            return "Perdesaan"
        return None

    df_long["Segmentasi Wilayah"] = df_long["KolomAsal"].apply(map_segmentasi)

    # Pastikan Presentase numerik
    df_long["Presentase"] = pd.to_numeric(df_long["Presentase"], errors="coerce")

    # Tambahkan Tahun
    df_long["Tahun"] = tahun

    # Pilih urutan kolom final
    df_long = df_long[["Provinsi", "Segmentasi Wilayah", "Tahun", "Presentase"]]
    return df_long


In [ ]:
file_per_tahun = {
    2020: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2020.csv",
    2021: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2021.csv",
    2022: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2022.csv",
    2023: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2023.csv",
    2024: "Persentase Rumah Tangga yang Memiliki_Menguasai  Telepon Tetap Kabel Menurut Provinsi dan Klasifikasi Daerah, 2024.csv",
}

df_list = []
for thn, nama_file in file_per_tahun.items():
    df_list.append(bersihkan_telepon(nama_file, thn))

df_telpon = pd.concat(df_list, ignore_index=True)

# Urutkan: Provinsi -> Segmentasi Wilayah -> Tahun (terbaru dulu)
df_telpon = df_telpon.sort_values(
    ["Provinsi", "Segmentasi Wilayah", "Tahun"],
    ascending=[True, True, False]
)

df_telpon.head(20)


,Provinsi,Segmentasi Wilayah,Tahun,Presentase
